# Maintainer's Copilot — DistilBERT Classifier (Colab T4)

Fine-tunes `distilbert-base-uncased` for 4-class GitHub issue triage:  
`bug` / `feature` / `docs` / `question`

**Before running:** upload your `.env` file to the Colab session root (`/content/.env`).
All secrets (MinIO credentials) are loaded from that file — nothing is hard-coded.

**Hardware:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 2 — Install dependencies
!pip install --quiet minio transformers datasets torch scikit-learn python-dotenv

In [ ]:
# Cell 3 — Load environment and set run ID
import os
import uuid
from pathlib import Path

from dotenv import load_dotenv

# In Colab, upload your .env to /content/.env before running.
env_path = Path("/content/.env")
if not env_path.exists():
    raise FileNotFoundError(
        "Upload your .env file to /content/.env before running this notebook."
    )

load_dotenv(dotenv_path=env_path, override=True)

MINIO_ENDPOINT = os.environ["MINIO_LOCAL_ENDPOINT"]
MINIO_ACCESS_KEY = os.environ["MINIO_ACCESS_KEY"]
MINIO_SECRET_KEY = os.environ["MINIO_SECRET_KEY"]
MINIO_BUCKET = os.environ["MINIO_BUCKET"]

RUN_ID = str(uuid.uuid4())
print(f"Run ID: {RUN_ID}")

In [ ]:
# Cell 4 — MinIO client setup and connection verification
from minio import Minio

endpoint = MINIO_ENDPOINT.removeprefix("http://").removeprefix("https://")
secure = not (endpoint.startswith("localhost") or endpoint.startswith("127."))

minio_client = Minio(
    endpoint,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=secure,
)

buckets = minio_client.list_buckets()
bucket_names = [b.name for b in buckets]
print(f"Connected to MinIO. Buckets: {bucket_names}")
assert MINIO_BUCKET in bucket_names, f"Bucket '{MINIO_BUCKET}' not found!"

In [ ]:
# Cell 5 — Download train.jsonl and val.jsonl from MinIO
import json
from io import BytesIO

def download_jsonl(bucket: str, key: str) -> list[dict]:
    """Download a JSONL object from MinIO and parse into a list of dicts."""
    response = minio_client.get_object(bucket, key)
    raw = response.read().decode("utf-8")
    return [json.loads(line) for line in raw.splitlines() if line.strip()]

print("Downloading splits from MinIO …")
train_data = download_jsonl(MINIO_BUCKET, "splits/v1/train.jsonl")
val_data   = download_jsonl(MINIO_BUCKET, "splits/v1/val.jsonl")

print(f"Train rows : {len(train_data)}")
print(f"Val rows   : {len(val_data)}")

In [ ]:
# Cell 6 — Class distribution and first 3 examples
from collections import Counter

LABEL2ID = {"bug": 0, "feature": 1, "docs": 2, "question": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def show_distribution(name: str, data: list[dict]) -> None:
    counts = Counter(row["label"] for row in data)
    total = len(data)
    print(f"\n{name} ({total} rows):")
    for cls in ("bug", "feature", "docs", "question"):
        n = counts.get(cls, 0)
        print(f"  {cls:<10} {n:>5}  ({100*n/total:.1f}%)")

show_distribution("train", train_data)
show_distribution("val",   val_data)

print("\n--- First 3 training examples ---")
for row in train_data[:3]:
    preview = row["text"][:120].replace("\n", " ")
    print(f"[{row['label']}] {preview} …")

In [ ]:
# Cell 7 — Tokenizer setup
from transformers import DistilBertTokenizerFast

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_batch(texts: list[str]) -> dict:
    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

# Smoke test
sample = tokenize_batch(["Test issue title"])
print(f"Tokenizer OK. Input IDs shape: {sample['input_ids'].shape}")

In [ ]:
# Cell 8 — IssueDataset (torch Dataset)
import torch
from torch.utils.data import Dataset

class IssueDataset(Dataset):
    """Tokenised GitHub issues dataset for DistilBERT fine-tuning."""

    def __init__(self, rows: list[dict], label2id: dict[str, int]) -> None:
        texts = [row["text"] for row in rows]
        labels = [label2id[row["label"]] for row in rows]

        encoding = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        self.input_ids = encoding["input_ids"]
        self.attention_mask = encoding["attention_mask"]
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }

print("Building datasets (this may take a minute) …")
train_dataset = IssueDataset(train_data, LABEL2ID)
val_dataset   = IssueDataset(val_data,   LABEL2ID)
print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset  : {len(val_dataset)} samples")

In [ ]:
# Cell 9 — Model setup
from transformers import DistilBertForSequenceClassification

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
# Cell 10 — Freeze all but last transformer block + classifier head
# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze: transformer block 5 (last of 6, zero-indexed)
for param in model.distilbert.transformer.layer[5].parameters():
    param.requires_grad = True

# Unfreeze: classification head
for param in model.pre_classifier.parameters():
    param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable parameters : {trainable:,}")
print(f"Frozen parameters    : {frozen:,}")
print(f"Trainable fraction   : {trainable / (trainable + frozen):.1%}")

In [ ]:
# Cell 11 — Training loop
import math
from io import BytesIO

from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup

# Hyperparameters (see model_card.md)
BATCH_SIZE    = 32
NUM_EPOCHS    = 5
LR            = 2e-5
WEIGHT_DECAY  = 0.01
WARMUP_STEPS  = 500
RANDOM_STATE  = 42

torch.manual_seed(RANDOM_STATE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps,
)

print(f"Steps per epoch : {len(train_loader)}")
print(f"Total steps     : {total_steps}")
print(f"Warmup steps    : {WARMUP_STEPS}")

def save_checkpoint_to_minio(epoch: int) -> None:
    buf = BytesIO()
    torch.save(model.state_dict(), buf)
    buf.seek(0)
    key = f"models/classifier/checkpoints/epoch_{epoch}_{RUN_ID}.pt"
    payload = buf.getvalue()
    minio_client.put_object(
        MINIO_BUCKET, key, BytesIO(payload), length=len(payload),
        content_type="application/octet-stream",
    )
    print(f"  Checkpoint saved → {key}")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        if (step + 1) % 50 == 0:
            avg = total_loss / (step + 1)
            print(f"  Epoch {epoch} step {step+1}/{len(train_loader)}  loss={avg:.4f}")

    avg_epoch_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch}/{NUM_EPOCHS} complete. Avg train loss: {avg_epoch_loss:.4f}")
    save_checkpoint_to_minio(epoch)

In [ ]:
# Cell 12 — Evaluate on validation set
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

def evaluate(loader: DataLoader, split_name: str) -> dict:
    model.eval()
    all_preds: list[int] = []
    all_labels: list[int] = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    label_names = [ID2LABEL[i] for i in range(len(ID2LABEL))]
    acc      = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")

    print(f"\n{'='*50}")
    print(f"{split_name} evaluation")
    print(f"{'='*50}")
    print(f"Accuracy   : {acc:.4f}")
    print(f"Macro-F1   : {f1_macro:.4f}")
    print("\nPer-class report:")
    print(classification_report(all_labels, all_preds, target_names=label_names))
    print("Confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(all_labels, all_preds))

    return {"accuracy": acc, "f1_macro": f1_macro}

val_metrics = evaluate(val_loader, "Validation")

In [ ]:
# Cell 13 — Final evaluation on test set
print("Downloading test split …")
test_data = download_jsonl(MINIO_BUCKET, "splits/v1/test.jsonl")
print(f"Test rows: {len(test_data)}")

test_dataset = IssueDataset(test_data, LABEL2ID)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_metrics = evaluate(test_loader, "Test (held-out)")

# CI gate thresholds from eval_thresholds.yaml
CI_ACC_THRESHOLD = 0.70
CI_F1_THRESHOLD  = 0.65

acc_pass = test_metrics["accuracy"] >= CI_ACC_THRESHOLD
f1_pass  = test_metrics["f1_macro"] >= CI_F1_THRESHOLD

print(f"\nCI gate check:")
print(f"  accuracy >= {CI_ACC_THRESHOLD}: {'PASS' if acc_pass else 'FAIL'}  ({test_metrics['accuracy']:.4f})")
print(f"  f1_macro >= {CI_F1_THRESHOLD}:  {'PASS' if f1_pass  else 'FAIL'}  ({test_metrics['f1_macro']:.4f})")

if not (acc_pass and f1_pass):
    print("\nWARNING: Model does not meet CI thresholds. Investigate before submitting.")

In [ ]:
# Cell 14 — Upload final weights to MinIO, compute SHA-256, print run summary
import hashlib

print("Saving final weights …")
weights_buf = BytesIO()
torch.save(model.state_dict(), weights_buf)
weights_bytes = weights_buf.getvalue()

weights_sha256 = hashlib.sha256(weights_bytes).hexdigest()
weights_key    = "models/classifier/weights.pt"

minio_client.put_object(
    MINIO_BUCKET,
    weights_key,
    BytesIO(weights_bytes),
    length=len(weights_bytes),
    content_type="application/octet-stream",
)

print("\n" + "="*60)
print("TRAINING COMPLETE — copy these values into model_card.md")
print("="*60)
print(f"Run ID         : {RUN_ID}")
print(f"Weights SHA-256: {weights_sha256}")
print(f"MinIO path     : {weights_key}")
print(f"Val  accuracy  : {val_metrics['accuracy']:.4f}")
print(f"Val  macro-F1  : {val_metrics['f1_macro']:.4f}")
print(f"Test accuracy  : {test_metrics['accuracy']:.4f}")
print(f"Test macro-F1  : {test_metrics['f1_macro']:.4f}")
print("="*60)

## Next steps

1. Copy the **SHA-256** and **Run ID** printed by Cell 14 into `model_card.md`
   in your VS Code project (the `Weights SHA-256` and `Run ID` fields).

2. Verify the model meets the CI thresholds from Cell 13 (`accuracy >= 0.70`,
   `macro-F1 >= 0.65`). If not, consider:
   - Unfreezing one more transformer block
   - Increasing training epochs
   - Investigating class imbalance in the label mapping

3. The weights are now at `models/classifier/weights.pt` in MinIO. The
   `bootcheck.py` in the API service will verify this path and its SHA-256
   against `model_card.md` at startup.

4. Proceed to **Phase 5** — classical ML baseline + LLM zero-shot baseline
   comparison on the same test split.